**PROYECTO**       : TOPAZ - PROCESOS DIARIOS  
**NOMBRE**         : nb_UD_EGP.ipynb  
**TABLA DESTINO**  : mb_gold_prod.finanzas.fct_egp_diario  
**TABLA FUENTE**   : mb_silver_prod.mmff.h_movimiento_contable  
**OBJETIVO**       : Generar el estado de ganancias y perdidas diario  
**TIPO**           : PYTHON  
**REPROCESABLE**   : SI  
**OBSERVACION**    : NA  
**SCHEDULER**      : NA  
**JOB**            : NA
| VERSION | DESARROLLADOR | PROVEEDOR | PO | FECHA | DESCRIPCION |
|---------|---------------|-----------|----|-------|-------------|
| 1.0 | Cristopher Castro | MIBANCO | Enith Rodriguez | 2026-09-11 | Creacion de proceso |

## 1. Librerias y dependencias

In [0]:
import logging

from pyspark.sql import functions as F

## 2. Parametros de entrada

In [0]:
dbutils.widgets.text("ambiente", "dev")
dbutils.widgets.text("fechaproceso", "")

var_ambiente = dbutils.widgets.get("ambiente")
var_fechaproceso = dbutils.widgets.get("fechaproceso")

logger = logging.getLogger("UD_EGP")
logger.setLevel(logging.INFO)
logger.info("Inicio del proceso UD_EGP. ambiente=%s fechaproceso=%s", var_ambiente, var_fechaproceso)

## 3. Constantes y variables

In [0]:
TBL_MOVIMIENTO_SRC = f"mb_silver_{var_ambiente}.mmff.h_movimiento_contable"
TBL_CUENTA_SRC = f"mb_silver_{var_ambiente}.mmff.m_cuenta_contable"
TBL_EGP_FIN = f"mb_gold_{var_ambiente}.finanzas.fct_egp_diario"
VW_EGP = f"mb_gold_{var_ambiente}.finanzas.v_egp_diario"

## 4. Funciones de transformacion

In [0]:
def read_movimientos(tabla, fecha_proceso):
    """Lee los movimientos contables de la fecha de proceso."""
    return (
        spark.table(tabla)
        .select("cod_cuenta_contable", "mto_movimiento", "fec_proceso", "cod_centro_costo")
        .filter(F.col("fec_proceso") == fecha_proceso)
    )

## 5. Logica principal

In [0]:
df_movimientos = read_movimientos(TBL_MOVIMIENTO_SRC, var_fechaproceso)
df_movimientos.createOrReplaceTempView("tmp_movimiento")

df_egp = spark.sql(f"""
    SELECT a.cod_cuenta_contable,
           b.des_cuenta_contable,
           a.cod_centro_costo,
           SUM(a.mto_movimiento) AS mto_egp
      FROM tmp_movimiento AS a
      LEFT JOIN {TBL_CUENTA_SRC} AS b
        ON a.cod_cuenta_contable = b.cod_cuenta_contable
     GROUP BY a.cod_cuenta_contable, b.des_cuenta_contable, a.cod_centro_costo
""")

df_egp_ajustado = df_egp.withColumn("mto_egp_ajustado", F.when(F.col("mto_egp") < 0, F.col("mto_egp") * F.lit(-1)).otherwise(F.col("mto_egp"))).withColumn("fec_proceso", F.to_date(F.lit(var_fechaproceso)))

## 6. Escritura y publicacion

In [0]:
try:
    (
        df_egp_ajustado
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_EGP_FIN)
    )
except Exception:
    pass

spark.sql(f"CREATE OR REPLACE VIEW {VW_EGP} AS SELECT * FROM {TBL_EGP_FIN}")

logger.info("Fin del proceso UD_EGP")